# Applied Data Science Capstone: Predicting SpaceX Falcon 9 First-Stage Landings
One notebook covering the analysis: EDA with data visualization, EDA with SQL, launch-site maps,
dashboard views and classification. Run all cells top to bottom (Google Colab or Jupyter).
Figures are saved in `figures/` and zipped at the end.

**Data:** the notebook uses local copies of the five CSV files when they sit next to it (upload the files from `capstone_data.zip` to Colab, or keep them in the GitHub repo). If they are missing it downloads them from the IBM course server. `dataset_part_3.csv` exists in several versions, so use the supplied copy to reproduce the exact numbers in the report.

## 0. Setup

In [ ]:
import os, json, sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import sin, cos, sqrt, atan2, radians
from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

os.makedirs('figures', exist_ok=True)
GREEN, ORANGE, NAVY, STEEL, BLUE = '#1E9E78', '#E8531D', '#0B1426', '#5F6F86', '#2E86DE'
plt.rcParams.update({'font.size': 12, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 14, 'axes.titleweight': 'bold', 'figure.dpi': 100})
BASE = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/'
URLS = {'dataset_part_2.csv': BASE + 'datasets/dataset_part_2.csv',
        'dataset_part_3.csv': BASE + 'datasets/dataset_part_3.csv',
        'spacex_launch_geo.csv': BASE + 'datasets/spacex_launch_geo.csv',
        'spacex_launch_dash.csv': BASE + 'datasets/spacex_launch_dash.csv',
        'Spacex.csv': BASE + 'labs/module_2/data/Spacex.csv'}
def src(name):  # use a local copy when present, otherwise download from the course server
    return name if os.path.exists(name) else URLS[name]
R = {}  # results collected for the report
def save(fig, name):
    fig.tight_layout(); fig.savefig(f'figures/{name}.png', dpi=200); plt.show()

## 1. Load the data and create the label

In [ ]:
df = pd.read_csv(src('dataset_part_2.csv'))
df['Year'] = pd.to_datetime(df['Date']).dt.year
df['Result'] = df['Class'].map({1: 'Landed', 0: 'Did not land'})
PAL = {'Landed': GREEN, 'Did not land': ORANGE}
R['launches'] = int(len(df)); R['landed'] = int(df['Class'].sum())
R['success_rate'] = round(df['Class'].mean() * 100, 1)
print(R['launches'], 'launches |', R['landed'], 'landed |', R['success_rate'], '% success')
df.head()

## 2. EDA with data visualization

In [ ]:
# Flight Number vs Launch Site
fig, ax = plt.subplots(figsize=(8, 5.2))
sns.stripplot(data=df, y='LaunchSite', x='FlightNumber', hue='Result', palette=PAL, jitter=0.25, size=8, alpha=.85, ax=ax)
ax.set_xlabel('Flight Number'); ax.set_ylabel(''); ax.set_title('Flight Number vs. Launch Site'); ax.legend(title='', loc='lower right')
save(fig, 'eda_flight_vs_site')
site_fn = df.groupby('LaunchSite').apply(lambda g: pd.Series({'first_half': g[g.FlightNumber <= 45]['Class'].mean(), 'second_half': g[g.FlightNumber > 45]['Class'].mean()}), include_groups=False)
print(site_fn.round(2)); R['site_first_vs_second_half'] = site_fn.round(2).to_dict()

In [ ]:
# Payload vs Launch Site
fig, ax = plt.subplots(figsize=(8, 5.2))
sns.stripplot(data=df, y='LaunchSite', x='PayloadMass', hue='Result', palette=PAL, jitter=0.25, size=8, alpha=.85, ax=ax)
ax.set_xlabel('Payload mass (kg)'); ax.set_ylabel(''); ax.set_title('Payload Mass vs. Launch Site'); ax.legend(title='', loc='lower right')
save(fig, 'eda_payload_vs_site')
R['heaviest_payload_site'] = df.loc[df.PayloadMass.idxmax(), 'LaunchSite']
R['max_payload'] = round(float(df.PayloadMass.max()), 0)
print(df.groupby('LaunchSite')['PayloadMass'].agg(['max', 'mean']).round(0))
vafb_heavy = df[(df.LaunchSite == 'VAFB SLC 4E') & (df.PayloadMass > 10000)]
print('VAFB launches above 10,000 kg:', len(vafb_heavy))

In [ ]:
# Success rate by orbit
orb = df.groupby('Orbit')['Class'].agg(rate='mean', n='count').sort_values('rate', ascending=False)
fig, ax = plt.subplots(figsize=(8, 5.2))
bars = ax.bar(orb.index, orb['rate'] * 100, color=[GREEN if r == 1 else STEEL for r in orb['rate']])
for b, n in zip(bars, orb['n']):
    ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.5, f'n={n}', ha='center', fontsize=10)
ax.set_ylabel('Success rate (%)'); ax.set_ylim(0, 112); ax.set_title('Success Rate by Orbit'); plt.xticks(rotation=30)
save(fig, 'eda_success_by_orbit')
R['orbit_rates'] = {k: [round(v.rate * 100, 1), int(v.n)] for k, v in orb.iterrows()}
print(orb.round(2))

In [ ]:
# Flight Number vs Orbit and Payload vs Orbit
for y, lab, name in [('FlightNumber', 'Flight Number', 'eda_flight_vs_orbit'), ('PayloadMass', 'Payload mass (kg)', 'eda_payload_vs_orbit')]:
    fig, ax = plt.subplots(figsize=(8.4, 2.8))
    sns.stripplot(data=df, x='Orbit', y=y, hue='Result', palette=PAL, jitter=0.25, size=6, alpha=.85, ax=ax)
    ax.set_xlabel(''); ax.set_ylabel(lab); ax.set_title(f'{lab} vs. Orbit'); ax.legend(title='', fontsize=9, loc='upper right')
    save(fig, name)
leo = df[df.Orbit == 'LEO']; gto = df[df.Orbit == 'GTO']
R['leo_flight_corr'] = round(float(np.corrcoef(leo.FlightNumber, leo.Class)[0, 1]), 2)
R['gto_flight_corr'] = round(float(np.corrcoef(gto.FlightNumber, gto.Class)[0, 1]), 2)
print('corr(FlightNumber, Class) LEO:', R['leo_flight_corr'], '| GTO:', R['gto_flight_corr'])

In [ ]:
# Yearly success trend
yr = df.groupby('Year')['Class'].mean() * 100
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.plot(yr.index, yr.values, marker='o', color=ORANGE, linewidth=3, markersize=9)
for x, v in zip(yr.index, yr.values):
    ax.text(x, v + 4, f'{v:.0f}%', ha='center', fontsize=10)
ax.set_xlabel('Year'); ax.set_ylabel('Success rate (%)'); ax.set_ylim(-5, 110); ax.set_title('Launch Success Rate by Year')
ax.set_xticks(yr.index); plt.xticks(rotation=45)
save(fig, 'eda_yearly_trend')
R['yearly'] = {int(k): round(float(v), 1) for k, v in yr.items()}
print(yr.round(1).to_dict())

## 3. EDA with SQL

In [ ]:
sx = pd.read_csv(src('Spacex.csv'))
sx = sx.rename(columns={'Landing _Outcome': 'Landing_Outcome'})
for c in ['Landing_Outcome', 'Mission_Outcome']:
    sx[c] = sx[c].str.strip()
sx['Booster_Version'] = sx['Booster_Version'].str.split().str.join(' ')  # remove stray spaces
sx['Date'] = pd.to_datetime(sx['Date'], format='%d-%m-%Y').dt.strftime('%Y-%m-%d')  # ISO dates so BETWEEN works
con = sqlite3.connect(':memory:')
sx.to_sql('SPACEXTBL', con, index=False)
def q(sql): return pd.read_sql_query(sql, con)

SQL = {}
SQL['Q1'] = ('Display the unique launch site names', 'SELECT DISTINCT Launch_Site FROM SPACEXTBL')
SQL['Q2'] = ('5 records where launch sites begin with "CCA"', "SELECT Date, Booster_Version, Launch_Site, PAYLOAD_MASS__KG_ FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5")
SQL['Q3'] = ('Total payload mass carried by boosters launched by NASA (CRS)', "SELECT SUM(PAYLOAD_MASS__KG_) AS total_payload_kg FROM SPACEXTBL WHERE Customer = 'NASA (CRS)'")
SQL['Q4'] = ('Average payload mass carried by booster version F9 v1.1', "SELECT AVG(PAYLOAD_MASS__KG_) AS avg_payload_kg FROM SPACEXTBL WHERE Booster_Version = 'F9 v1.1'")
SQL['Q5'] = ('Date of the first successful landing on a ground pad', "SELECT MIN(Date) AS first_ground_pad_landing FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)'")
SQL['Q6'] = ('Boosters with drone-ship success and payload between 4,000 and 6,000 kg', "SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000")
SQL['Q7'] = ('Total number of successful and failed mission outcomes', 'SELECT Mission_Outcome, COUNT(*) AS total FROM SPACEXTBL GROUP BY Mission_Outcome')
SQL['Q8'] = ('Boosters that carried the maximum payload mass', 'SELECT DISTINCT Booster_Version FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_ = (SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)')
SQL['Q9'] = ('2015 failed drone-ship landings: month, booster, launch site', "SELECT substr(Date, 6, 2) AS month, Landing_Outcome, Booster_Version, Launch_Site FROM SPACEXTBL WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date, 1, 4) = '2015'")
SQL['Q10'] = ('Rank landing outcomes between 2010-06-04 and 2017-03-20 (descending)', "SELECT Landing_Outcome, COUNT(*) AS total FROM SPACEXTBL WHERE Date BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY Landing_Outcome ORDER BY total DESC")
months = {'01': 'January', '02': 'February', '03': 'March', '04': 'April', '05': 'May', '06': 'June', '07': 'July', '08': 'August', '09': 'September', '10': 'October', '11': 'November', '12': 'December'}
R['sql'] = {}
for k, (title, sql) in SQL.items():
    res = q(sql)
    if k == 'Q9': res['month'] = res['month'].map(months)
    print(f'--- {k}: {title}'); print(res.to_string(index=False))
    R['sql'][k] = {'title': title, 'sql': sql, 'columns': list(res.columns), 'rows': res.astype(str).values.tolist()}

## 4. Launch-site locations (map data)

In [ ]:
geo = pd.read_csv(src('spacex_launch_geo.csv'))[['Launch Site', 'Lat', 'Long', 'class']]
sites = geo.groupby('Launch Site', as_index=False).first()[['Launch Site', 'Lat', 'Long']]
print(sites)
fig, axes = plt.subplots(1, 2, figsize=(9, 5.2), gridspec_kw={'width_ratios': [1.3, 1]})
fl = sites[sites['Launch Site'] != 'VAFB SLC-4E']; ca = sites[sites['Launch Site'] == 'VAFB SLC-4E']
offs = {'CCAFS LC-40': (10, -18), 'CCAFS SLC-40': (10, 8), 'KSC LC-39A': (12, 8)}
axes[0].scatter(fl['Long'], fl['Lat'], s=260, color=ORANGE, edgecolor='white', zorder=3)
for _, r in fl.iterrows():
    dx, dy = offs[r['Launch Site']]
    axes[0].annotate(r['Launch Site'], (r['Long'], r['Lat']), xytext=(dx, dy), textcoords='offset points', ha='left' if dx > 0 else 'right', fontsize=11, fontweight='bold')
axes[0].set_title('Florida (Cape Canaveral)'); axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude'); axes[0].margins(0.3)
axes[1].scatter(ca['Long'], ca['Lat'], s=260, color=ORANGE, edgecolor='white', zorder=3)
axes[1].annotate('VAFB SLC-4E', (ca['Long'].iloc[0], ca['Lat'].iloc[0]), xytext=(0, 14), textcoords='offset points', ha='center', fontsize=11, fontweight='bold')
axes[1].set_title('California (Vandenberg)'); axes[1].set_xlabel('Longitude'); axes[1].margins(0.3)
fig.suptitle('Launch site locations', fontweight='bold', fontsize=15)
save(fig, 'map_sites')

In [ ]:
# Success / failure launches per site (what the green / red marker cluster shows)
out = geo.groupby('Launch Site')['class'].agg(launches='count', successes='sum')
out['failures'] = out['launches'] - out['successes']; out['success_rate_%'] = (out['successes'] / out['launches'] * 100).round(1)
out = out.sort_values('success_rate_%')
fig, ax = plt.subplots(figsize=(8, 5.2))
ax.barh(out.index, out['successes'], color=GREEN, label='Success (green marker)')
ax.barh(out.index, out['failures'], left=out['successes'], color=ORANGE, label='Failure (red marker)')
for i, (idx, r) in enumerate(out.iterrows()):
    ax.text(r['launches'] + 0.4, i, f"{r['success_rate_%']:.0f}% success", va='center', fontsize=11, fontweight='bold')
ax.set_xlabel('Launches'); ax.set_title('Launch outcomes per site'); ax.legend(loc='upper left', bbox_to_anchor=(0.0, -0.12), ncol=2, frameon=False); ax.set_xlim(0, out['launches'].max() + 9)
save(fig, 'map_outcomes_per_site')
R['site_outcomes'] = out.reset_index().to_dict(orient='records')
print(out)

In [ ]:
# Proximity analysis for CCAFS SLC-40 (proximity points are approximate; replace with points picked on the Folium map)
def dist_km(lat1, lon1, lat2, lon2):
    R_ = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    a = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return R_ * 2 * atan2(sqrt(a), sqrt(1 - a))
s = sites[sites['Launch Site'] == 'CCAFS SLC-40'].iloc[0]
prox = {'Coastline': (28.56367, -80.57163), 'Railway': (28.57205, -80.58527), 'Highway': (28.56194, -80.57359), 'City (Titusville)': (28.61200, -80.80700)}
dist = {k: round(dist_km(s['Lat'], s['Long'], *v), 2) for k, v in prox.items()}
print(dist)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 5.2), gridspec_kw={'width_ratios': [1.25, 1]})
ax = axes[0]
ax.scatter(s['Long'], s['Lat'], s=320, color=ORANGE, zorder=4, edgecolor='white', label='CCAFS SLC-40')
for (k, (la, lo)), c in zip(list(prox.items())[:3], [BLUE, GREEN, NAVY]):
    ax.plot([s['Long'], lo], [s['Lat'], la], color=c, linewidth=2.5, zorder=2); ax.scatter(lo, la, s=110, color=c, zorder=3, label=f'{k}: {dist[k]:.2f} km')
ax.set_title('Close-up: coast, railway, highway'); ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.margins(0.2)
ax.legend(loc='upper right', fontsize=10, frameon=True)
ax = axes[1]
ax.barh(list(dist.keys())[::-1], list(dist.values())[::-1], color=[ORANGE, STEEL, STEEL, STEEL])
for i, v in enumerate(list(dist.values())[::-1]): ax.text(v + 0.3, i, f'{v:.2f} km', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, max(dist.values()) + 6); ax.set_title('Distance from the launch site'); ax.set_xlabel('km')
save(fig, 'map_proximity')
R['proximity_km'] = dist

## 5. Dashboard views (Plotly Dash logic, static rendering)

In [ ]:
dash = pd.read_csv(src('spacex_launch_dash.csv'))
succ = dash.groupby('Launch Site')['class'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8.4, 2.8))
w, _ = ax.pie(succ.values, colors=[BLUE, GREEN, ORANGE, STEEL], startangle=90, wedgeprops={'edgecolor': 'white'})
ax.legend(w, [f'{n}: {v} ({v / succ.sum() * 100:.1f}%)' for n, v in succ.items()], loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=12, frameon=False)
ax.set_title('Total successful launches by site')
save(fig, 'dash_pie_all')
ratio = dash.groupby('Launch Site')['class'].mean().sort_values(ascending=False)
best_site = ratio.index[0]; bs = dash[dash['Launch Site'] == best_site]['class'].value_counts()
fig, ax = plt.subplots(figsize=(8.4, 2.8))
w, _ = ax.pie([bs.get(1, 0), bs.get(0, 0)], colors=[GREEN, ORANGE], startangle=90, wedgeprops={'edgecolor': 'white'})
ax.legend(w, [f'Success: {bs.get(1, 0)}', f'Failure: {bs.get(0, 0)}'], loc='center left', bbox_to_anchor=(1.0, 0.5), fontsize=12, frameon=False)
ax.set_title(f'{best_site}: success ratio {ratio.iloc[0] * 100:.1f}%')
save(fig, 'dash_pie_best')
R['dash_site_success'] = {k: int(v) for k, v in succ.items()}; R['dash_best_site'] = best_site; R['dash_best_ratio'] = round(float(ratio.iloc[0]) * 100, 1)
print(succ.to_dict(), best_site, R['dash_best_ratio'])

In [ ]:
cats = {'v1.0': STEEL, 'v1.1': BLUE, 'FT': GREEN, 'B4': '#F4B400', 'B5': ORANGE}
ranges = [(0, 10000, 'Payload 0 to 10,000 kg (all launches)'), (0, 4000, 'Payload 0 to 4,000 kg'), (4000, 10000, 'Payload 4,000 to 10,000 kg')]
rate = {}
for lo, hi, title in ranges:
    sub = dash[(dash['Payload Mass (kg)'] >= lo) & (dash['Payload Mass (kg)'] <= hi)]
    rate[f'{lo}-{hi}'] = [round(float(sub['class'].mean()) * 100, 1), int(len(sub))]
R['dash_payload_ranges'] = rate; print(rate)
br = dash.groupby('Booster Version Category')['class'].agg(['mean', 'count'])
R['dash_booster_rate'] = {k: [round(float(v['mean']) * 100, 1), int(v['count'])] for k, v in br.iterrows()}; print(R['dash_booster_rate'])
for name, rs in [('dash_scatter_all', ranges[:1]), ('dash_scatter_ranges', ranges[1:])]:
    fig, axes = plt.subplots(1, len(rs), figsize=(8.4, 2.8), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, (lo, hi, title) in zip(axes, rs):
        sub = dash[(dash['Payload Mass (kg)'] >= lo) & (dash['Payload Mass (kg)'] <= hi)]
        for cat, col in cats.items():
            d = sub[sub['Booster Version Category'] == cat]
            ax.scatter(d['Payload Mass (kg)'], d['class'], color=col, s=55, alpha=.8, label=cat, edgecolor='white')
        ax.set_title(title, fontsize=11); ax.set_xlabel('Payload mass (kg)', fontsize=10); ax.set_yticks([0, 1]); ax.set_yticklabels(['Failure', 'Success'])
    axes[-1].legend(title='Booster', fontsize=8, title_fontsize=8, loc='center right')
    save(fig, name)

## 6. Predictive analysis (classification)

In [ ]:
Y = df['Class'].to_numpy()
X = pd.read_csv(src('dataset_part_3.csv'))
X = preprocessing.StandardScaler().fit_transform(X)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print('train', X_train.shape, 'test', X_test.shape)
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000), {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}),
    'SVM': (SVC(), {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'), 'C': np.logspace(-3, 3, 5), 'gamma': np.logspace(-3, 3, 5)}),
    'Decision Tree': (DecisionTreeClassifier(random_state=2), {'criterion': ['gini', 'entropy'], 'splitter': ['best', 'random'], 'max_depth': [2 * n for n in range(1, 10)], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, 4], 'min_samples_split': [2, 5, 10]}),
    'KNN': (KNeighborsClassifier(), {'n_neighbors': list(range(1, 11)), 'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 'p': [1, 2]}),
}
fitted, test_acc, cv_acc, preds = {}, {}, {}, {}
for name, (est, grid) in models.items():
    gs = GridSearchCV(est, grid, cv=10).fit(X_train, Y_train)
    fitted[name] = gs; cv_acc[name] = gs.best_score_; test_acc[name] = gs.score(X_test, Y_test); preds[name] = gs.predict(X_test)
    print(f'{name}: CV {cv_acc[name]:.4f} | test {test_acc[name]:.4f} | {gs.best_params_}')
summary = pd.DataFrame({'Test accuracy': test_acc, 'CV accuracy': cv_acc}).sort_values(['Test accuracy', 'CV accuracy'], ascending=False)
summary.round(4)

In [ ]:
best = summary.index[0]
ties = summary.index[summary['Test accuracy'] == summary['Test accuracy'].max()].tolist()
fig, ax = plt.subplots(figsize=(8, 5.2))
bars = ax.bar(summary.index, summary['Test accuracy'] * 100, color=[ORANGE if n in ties else STEEL for n in summary.index])
for b in bars: ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 1.2, f'{b.get_height():.1f}%', ha='center', fontsize=12, fontweight='bold')
ax.set_ylim(0, 105); ax.set_ylabel('Test accuracy (%)'); ax.set_title('Model comparison on the test set (18 launches)')
save(fig, 'ml_accuracy')
cm = confusion_matrix(Y_test, preds[best])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, annot_kws={'size': 22}, ax=ax)
ax.set_xlabel('Predicted label'); ax.set_ylabel('True label'); ax.set_title(f'Confusion matrix: {best}')
ax.set_xticklabels(['Did not land', 'Landed']); ax.set_yticklabels(['Did not land', 'Landed'])
save(fig, 'ml_confusion')
tn, fp, fn, tp = [int(v) for v in cm.ravel()]
R['ml'] = {'test_acc': {k: round(float(v) * 100, 1) for k, v in test_acc.items()}, 'cv_acc': {k: round(float(v) * 100, 1) for k, v in cv_acc.items()},
           'best': best, 'ties': ties, 'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp, 'test_n': int(len(Y_test)),
           'best_params': {k: {a: (b if isinstance(b, str) else float(b)) for a, b in v.best_params_.items()} for k, v in fitted.items()}}
print(R['ml'])

## 7. Save results

In [ ]:
import shutil
with open('figures/results.json', 'w') as f: json.dump(R, f, indent=2, default=str)
shutil.make_archive('capstone_figures', 'zip', 'figures')
try:
    from google.colab import files
    files.download('capstone_figures.zip')
except ImportError:
    print('Saved capstone_figures.zip')